In [16]:
import json
import os
import pandas as pd
import glob
from transformers import AutoTokenizer
import re
import json
from dotenv import load_dotenv
from copy import deepcopy
from tqdm import tqdm
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [17]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result

In [18]:
dataset_type = f'hoasa_hotel'
dataset_folder = 'mvp_aos'
data_path = f'dataset/{dataset_type}/indo/{dataset_folder}/train.json'
with open(data_path, 'r') as f:
    data = json.load(f)
data[0]

{'sentence_id': 0,
 'instance_id': 0,
 'input': 'kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil . [A] [O] [S]',
 'target': '[A] ac [O] tidak berfungsi optimal [S] negative [SSEP] [A] wifi koneksi [O] kurang stabil [S] negative',
 'element_order': 'aos',
 'task_elements': 'aos',
 'dataset_type': 'hotel_reviews'}

In [19]:
# First check
mistakes = {
	"hoasa": {
		'a': [],
		'o': [],
		's': [],
	},
	"hotel_reviews": {
		'a': [],
		'o': [],
		's': [],
	}
}
for instance in data:
	parsed_target = parse_absa_string(instance['target'])
	input_text = instance['input']

	for triplet in parsed_target:
		triplet_true = True
		aspect = triplet['A']
		if aspect not in input_text and aspect.lower() != 'null':
			print(f"Aspect Sentence ID {instance['sentence_id']}:\nAspect '{aspect}' not found in input: {input_text}")
			triplet_true = False
			mistakes[instance['dataset_type']]['a'].append(instance['sentence_id'])
		
		opinion = triplet['O']
		if opinion not in input_text and opinion.lower() != 'null':
			print(f"Opinion Sentence ID {instance['sentence_id']}:\nOpinion '{opinion}' not found in input: {input_text}")
			triplet_true = False
			mistakes[instance['dataset_type']]['o'].append(instance['sentence_id'])
		
		sentiment = triplet['S']
		if sentiment.lower() not in ['positive', 'negative']:
			print(f"Sentiment Sentence ID {instance['sentence_id']}:\nSentiment '{sentiment}' is not valid in input: {input_text}")
			triplet_true = False
			mistakes[instance['dataset_type']]['s'].append(instance['sentence_id'])
		
		if not triplet_true:
			print(f"Full Target: {instance['target']}")
			print(instance['dataset_type'])
			print("--------------------------------------------------")

Aspect Sentence ID 368:
Aspect 'airnya' not found in input: sejauh ini baik sih , cuma penginapan nya gelap banget , cat lorong nya warna oren lalu lampu nya warna kuning jadi gelap banget , sempat seram juga sih setiap pulang malam tetapi lama lama iya biasa saja , terus setiap mandi air nya keluar kamar mandi jadi di depan kamar mandi basah semua , awal datang tidak di sediakan selimut dan keset kamar mandi . harus minta . [A] [O] [S]
Full Target: [A] penginapan nya [O] gelap banget [S] negative [SSEP] [A] penginapan nya [O] cat lorong nya warna oren lalu lampu nya warna kuning jadi gelap banget [S] negative [SSEP] [A] penginapan nya [O] sempat seram juga sih setiap pulang malam [S] negative [SSEP] [A] penginapan nya [O] tetapi lama lama iya biasa saja [S] positive [SSEP] [A] airnya [O] setiap mandi air nya keluar kamar mandi jadi di depan kamar mandi basah semua [S] negative [SSEP] [A] selimut [O] tidak di sediakan [S] negative [SSEP] [A] keset kamar mandi [O] tidak di sediakan [S] 

In [20]:
for key, value in mistakes.items():
	print(f"Mistakes in {key}:")
	for tag, ids in value.items():
		unique_ids = set(ids)
		print(f" {len(unique_ids)} {tag.upper()} mistakes in sentence IDs: {sorted(unique_ids)}")

Mistakes in hoasa:
 2 A mistakes in sentence IDs: [178, 464]
 5 O mistakes in sentence IDs: [464, 735, 1066, 1593, 2087]
 112 S mistakes in sentence IDs: [88, 109, 129, 154, 158, 171, 188, 200, 220, 230, 234, 268, 276, 279, 294, 374, 383, 419, 436, 463, 467, 473, 474, 492, 502, 505, 506, 523, 563, 621, 653, 666, 680, 738, 808, 818, 830, 836, 879, 894, 962, 964, 974, 978, 1008, 1019, 1036, 1044, 1078, 1083, 1087, 1102, 1109, 1138, 1140, 1141, 1157, 1176, 1180, 1188, 1199, 1200, 1236, 1245, 1246, 1259, 1264, 1312, 1316, 1347, 1359, 1360, 1378, 1379, 1389, 1392, 1400, 1415, 1439, 1440, 1443, 1444, 1448, 1476, 1495, 1508, 1514, 1528, 1540, 1568, 1575, 1600, 1612, 1618, 1674, 1699, 1750, 1783, 1824, 1844, 1849, 1892, 1940, 1983, 1991, 2005, 2060, 2061, 2085, 2088, 2109, 2133]
Mistakes in hotel_reviews:
 2 A mistakes in sentence IDs: [368, 2372]
 0 O mistakes in sentence IDs: []
 0 S mistakes in sentence IDs: []


In [21]:
# Second Check (word level match)
mistakes = {
	"hoasa": {
		'a': [],
		'o': [],
	},
	"hotel_reviews": {
		'a': [],
		'o': [],
	}
}
for instance in data:
	parsed_target = parse_absa_string(instance['target'])
	input_text = instance['input'].lower()
	input_text_words = input_text.split()

	for triplet in parsed_target:
		triplet_true = True
		aspect = triplet['A'].lower()
		if aspect != 'null':
			for word in aspect.split():
				if word not in input_text_words:
					print(f"Aspect Sentence ID {instance['sentence_id']}:\nAspect word '{word}' from aspect '{aspect}' not found in input: {input_text_words}")
					triplet_true = False
					mistakes[instance['dataset_type']]['a'].append(instance['sentence_id'])
		
		opinion = triplet['O'].lower()
		if opinion != 'null':
			for word in opinion.split():
				if word not in input_text_words:
					print(f"Opinion Sentence ID {instance['sentence_id']}:\nOpinion word '{word}' from opinion '{opinion}' not found in input: {input_text_words}")
					triplet_true = False
					mistakes[instance['dataset_type']]['o'].append(instance['sentence_id'])
		
		if not triplet_true:
			print(f"Full Target: {instance['target']}")
			print(instance['dataset_type'])
			print("--------------------------------------------------")

Opinion Sentence ID 225:
Opinion word 'safar' from opinion 'dekat dengan taman safar' not found in input: ['bersih', '.', 'nyaman', '.', 'ramah', '.', 'dekat', 'dengan', 'taman', 'safari', '.', '[a]', '[o]', '[s]']
Full Target: [A] null [O] bersih [S] positive [SSEP] [A] null [O] nyaman [S] positive [SSEP] [A] null [O] ramah [S] positive [SSEP] [A] null [O] dekat dengan taman safar [S] positive
hotel_reviews
--------------------------------------------------
Opinion Sentence ID 236:
Opinion word 'nyala' from opinion 'tidak nyala' not found in input: ['kamar', 'bocor', ',', 'air', 'panas', 'tidak', 'nyalah', ',', 'ac', 'tidak', 'dingin', ',', '.', '[a]', '[o]', '[s]']
Full Target: [A] kamar [O] bocor [S] negative [SSEP] [A] air panas [O] tidak nyala [S] negative [SSEP] [A] ac [O] tidak dingin [S] negative
hotel_reviews
--------------------------------------------------
Opinion Sentence ID 350:
Opinion word 'ngebut' from opinion 'ngebut' not found in input: ['nyaman', ',', 'bersih', ',',

In [22]:
for key, value in mistakes.items():
	print(f"Probable mistakes in {key}:")
	for tag, ids in value.items():
		unique_ids = set(ids)
		print(f"  {tag.upper()} mistakes in sentence IDs: {len(unique_ids)} {sorted(unique_ids)}")

Probable mistakes in hoasa:
  A mistakes in sentence IDs: 12 [64, 464, 749, 787, 957, 974, 986, 1222, 1386, 1500, 1879, 1978]
  O mistakes in sentence IDs: 16 [180, 405, 464, 541, 628, 735, 754, 891, 973, 1066, 1461, 1573, 1593, 1948, 1978, 2087]
Probable mistakes in hotel_reviews:
  A mistakes in sentence IDs: 8 [368, 447, 1022, 1144, 1425, 1489, 1942, 2372]
  O mistakes in sentence IDs: 19 [225, 236, 350, 361, 588, 1022, 1213, 1487, 1587, 1942, 2007, 2011, 2096, 2132, 2181, 2262, 2302, 2351, 2474]
